# Brightway 2.5 实践教学：饮用水与水瓶生产生命周期评价 (Practice 2: Bottled Water LCA)

---

## 教学目标与核心要点 (Learning Objectives)

欢迎进入 **Brightway 2.5** 进阶实操课堂！本项目严格基于 Brightway 官方核心教程 [Introduction to Brightway25](https://learn.brightway.dev/en/latest/content/chapters/BW25/BW25_introduction.html) 设计，以**瓶装饮用水（Bottled Water）**与**塑料水瓶生产（Water Bottle Packaging）**为典型工业案例，带领大家系统掌握 Brightway 2.5 的核心架构与前沿工程技能。

### Brightway 2 到 Brightway 2.5 的核心范式变革
1. **从“面向结构 (Structure-centric)”到“面向数据 (Data-centric)”**：
   - **BW2**：强依赖 SQLite/Peewee ORM 关系型数据库管理生命周期活动与图连接，矩阵构造过程黑盒化。
   - **BW2.5**：引入 `datapackages`、`bw_processing` 与 `matrix_utils`，将底层技术矩阵与特征化矩阵解耦为标准化的 NumPy 二进制数据包。运算速度提升数倍，支持无需数据库常驻的内存极速矩阵构建。
2. **全新 LCIA 计算接口与 MultiLCA 体系**：
   - 单工序计算重构为 `prepare_lca_inputs()` 与数据对象传递机制。
   - 全新 2024 版 `MultiLCA` 彻底废弃旧版字典模板配置，原生支持多功能单位与多环境影响类别的批量并行求解。
3. **更严格的数据质量控制与排错机制**：
   - 演示因单位书写差异（如 `piece` vs `unit`）导致的未匹配流排错（Unlinked Exchanges），掌握字段对齐技巧。
4. **零商业授权自洽教学保障 (Solution 1: Zero-License Background Model)**：
   - 官方教程依赖商业付费版 Ecoinvent 3.9.1 数据库。本教学演示包内置**自主模拟背景数据库与环境影响评价方法**，无需商业授权与网络依赖，解压即开即用！
5. **官方同款环境配置方案 (`data_collections/env_bw25.yml`)**：
   - 本教学包在 `data_collections/` 中内置了官方标准 Conda 依赖文件 `env_bw25.yml`，支持一键命令 `conda env create -f data_collections/env_bw25.yml` 100% 精准克隆官方环境，解压即开即用。

---

### 实操章节导航
- **第 1 步**：环境准备、Windows 求解器稳定性补丁与核心库验证
- **第 2 步**：BW2.5 项目架构与环境初始化 (`bw25-tuto`)
- **第 3 步**：零商业授权背景数据库与多指标评价方法构建
- **第 4 步**：前景清单 Excel 导入、策略流水线与数据库匹配
- **第 5 步**：教学专题：未链接流排错诊断与交互式修复演练
- **第 6 步**：单工序 LCIA 计算与 BW2.5 核心 API 剖析
- **第 7 步**：全新 2024 MultiLCA 批量计算与多指标对比分析
- **第 8 步**：底层矩阵解构、贡献度溯源与专业级多 Sheet Excel 报告导出
- **第 9 步**：蒙特卡洛不确定性分析与分布统计可视化


In [ ]:
# =============================================================
# 第 1 步：环境准备、Windows 求解器稳定性补丁与核心库验证
# =============================================================
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1.1 Windows 求解器稳定性关键补丁 (绕过 pypardiso 的 MKL 依赖缺失风险)
# -------------------------------------------------------------
import scipy.sparse.linalg
try:
    import bw2calc.lca_base
    import bw2calc.multi_lca
    # 强制将求解器指向 scipy.sparse.linalg.spsolve
    bw2calc.lca_base.spsolve = scipy.sparse.linalg.spsolve
    bw2calc.multi_lca.spsolve = scipy.sparse.linalg.spsolve
    # 禁用底层 C 级别 pypardiso 释放调用，杜绝 Windows 0xc06d007f 崩溃
    bw2calc.lca_base.pypardiso_solver = None
    bw2calc.lca_base.LCABase._delete_solver_state = lambda self: None
    print("[系统补丁] 已成功注入 Windows 求解器稳定性补丁 (使用 SciPy 稀疏求解器)！")
except Exception as e:
    print(f"[系统补丁] 注入异常: {e}")

# 1.2 核心科学计算与 LCA 依赖导入
# -------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import bw2data as bd
import bw2calc as bc
import bw2io as bi
import bw2analyzer as bwa
import bw_processing as bwp
import matrix_utils as mu

# 1.3 Matplotlib 中文显示与绘图风格设置 (持久化注入中文字体，防 Seaborn 重置覆盖)
# -------------------------------------------------------------
sns.set_theme(
    style="whitegrid",
    palette="muted",
    rc={
        'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Source Han Sans CN', 'DejaVu Sans'],
        'axes.unicode_minus': False,
        'figure.dpi': 120
    }
)
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Source Han Sans CN', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 1.4 定义自适应相对数据路径
# -------------------------------------------------------------
DATA_DIR = Path("data_collections")
assert DATA_DIR.exists(), f"未找到数据目录: {DATA_DIR.resolve()}"

print(f"\n[系统信息] 运行环境核查通过:")
print(f"   * Python 版本:    {sys.version.split()[0]}")
print(f"   * bw2data 版本:   {bd.__version__} (BW2.5 核心)")
print(f"   * bw2calc 版本:   {bc.__version__} (BW2.5 计算引擎)")
print(f"   * bw2io 版本:     {bi.__version__} (输入输出与转换)")
print(f"   * matrix_utils:   {mu.__version__}")
print(f"   * bw_processing:  {bwp.__version__}")
print(f"   * 数据源路径:     {DATA_DIR.resolve()}")


---

## 第 2 步：BW2.5 项目架构与环境初始化

在 Brightway 中，项目（Project）是所有数据库、评价方法、计算参数和计算缓存的顶层隔离容器。不同项目之间的数据完全独立。

在 Brightway 2.5 中，项目元数据依然保存在本地数据库中，但与 BW2 相比，底层存储更加轻量、解耦，并原生支持高效的多项目切换。


In [ ]:
# =============================================================
# 第 2 步：初始化 Brightway 2.5 项目 (bw25-tuto)
# =============================================================
PROJECT_NAME = "bw25-tuto"

# 为保证教学演示具有 100% 可重复性，如果已存在该项目则先重置
if PROJECT_NAME in bd.projects:
    print(f"检测到历史项目 [{PROJECT_NAME}]，正在执行重置以保证全新教学环境...")
    bd.projects.delete_project(PROJECT_NAME, delete_dir=True)

bd.projects.set_current(PROJECT_NAME)

print(f"[项目信息] 当前活跃 Brightway 项目: {bd.projects.current}")
print(f"[项目信息] 数据存储物理目录:    {bd.projects.dir}")
print(f"[项目信息] 当前已注册数据库列表:    {list(bd.databases)}")
print(f"[项目信息] 当前已注册评价方法数:    {len(bd.methods)}")


---

## 第 3 步：教学保障：零商业授权背景数据库构建 (Solution 1)

> **教学背景与方案设计**：
> 官方教程假定学习者拥有商业版 **Ecoinvent 3.9.1 (Cutoff)** 数据库授权，并在代码中直接引用 `ecoinvent-3.9.1-cutoff` 与 `ecoinvent-3.9.1-biosphere`。
> 现实中高校学生与初学者普遍无法直接获得昂贵的商业授权。
> 为此，我们在此实现 **零商业授权自洽解决方案 (Solution 1)**：在本地项目中自主构建轻量、真实且符合官方清单命名的背景数据库与 4 类主流 LCIA 特征化方法。学生无需配置任何外部商业许可证，即可 100% 顺畅执行官方所有代码！


In [ ]:
# =============================================================
# 第 3 步：构建零商业授权背景数据库与环境影响评价方法
# =============================================================

# 3.1 构建基本流环境数据库 (ecoinvent-3.9.1-biosphere)
# -------------------------------------------------------------
bio_db = bd.Database("ecoinvent-3.9.1-biosphere")
biosphere_data = {
    ('ecoinvent-3.9.1-biosphere', 'co2-fossil'): {
        'name': 'Carbon dioxide, fossil',
        'unit': 'kilogram',
        'type': 'emission',
        'categories': ('air', 'unspecified'),
    },
    ('ecoinvent-3.9.1-biosphere', 'ch4-fossil'): {
        'name': 'Methane, fossil',
        'unit': 'kilogram',
        'type': 'emission',
        'categories': ('air', 'unspecified'),
    },
    ('ecoinvent-3.9.1-biosphere', 'n2o'): {
        'name': 'Dinitrogen monoxide',
        'unit': 'kilogram',
        'type': 'emission',
        'categories': ('air', 'unspecified'),
    },
    ('ecoinvent-3.9.1-biosphere', 'water-well'): {
        'name': 'Water, well, in ground',
        'unit': 'cubic meter',
        'type': 'natural resource',
        'categories': ('natural resource', 'in water'),
    },
    ('ecoinvent-3.9.1-biosphere', 'so2'): {
        'name': 'Sulfur dioxide',
        'unit': 'kilogram',
        'type': 'emission',
        'categories': ('air', 'unspecified'),
    },
}
bio_db.write(biosphere_data)
print(f"[数据库构建] 基本流库已写入: {bio_db.name} (包含 {len(bio_db)} 个环境基本流)")

# 3.2 构建背景过程数据库 (ecoinvent-3.9.1-cutoff)
# 包含水瓶制造与饮用水供应链所需的 5 个核心上游工业工序
# -------------------------------------------------------------
bg_db = bd.Database("ecoinvent-3.9.1-cutoff")
cutoff_data = {
    ('ecoinvent-3.9.1-cutoff', 'pet-granulate'): {
        'name': 'polyethylene terephthalate production, granulate, bottle grade',
        'reference product': 'polyethylene terephthalate, granulate, bottle grade',
        'unit': 'kilogram',
        'location': 'GLO',
        'exchanges': [
            {'input': ('ecoinvent-3.9.1-cutoff', 'pet-granulate'), 'amount': 1.0, 'unit': 'kilogram', 'type': 'production'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 'amount': 2.15, 'unit': 'kilogram', 'type': 'biosphere'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'ch4-fossil'), 'amount': 0.005, 'unit': 'kilogram', 'type': 'biosphere'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'water-well'), 'amount': 0.020, 'unit': 'cubic meter', 'type': 'biosphere'},
        ]
    },
    ('ecoinvent-3.9.1-cutoff', 'blow-moulding'): {
        'name': 'blow moulding',
        'reference product': 'blow moulding',
        'unit': 'kilogram',
        'location': 'RER',
        'exchanges': [
            {'input': ('ecoinvent-3.9.1-cutoff', 'blow-moulding'), 'amount': 1.0, 'unit': 'kilogram', 'type': 'production'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 'amount': 0.85, 'unit': 'kilogram', 'type': 'biosphere'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'so2'), 'amount': 0.002, 'unit': 'kilogram', 'type': 'biosphere'},
        ]
    },
    ('ecoinvent-3.9.1-cutoff', 'tap-water'): {
        'name': 'tap water production',
        'reference product': 'tap water',
        'unit': 'kilogram',
        'location': 'Europe without Switzerland',
        'exchanges': [
            {'input': ('ecoinvent-3.9.1-cutoff', 'tap-water'), 'amount': 1.0, 'unit': 'kilogram', 'type': 'production'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 'amount': 0.0003, 'unit': 'kilogram', 'type': 'biosphere'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'water-well'), 'amount': 0.0012, 'unit': 'cubic meter', 'type': 'biosphere'},
        ]
    },
    ('ecoinvent-3.9.1-cutoff', 'transport-lorry'): {
        'name': 'market for transport, freight, lorry 16-32 metric ton, EURO5',
        'reference product': 'transport, freight, lorry 16-32 metric ton, EURO5',
        'unit': 'ton kilometer',
        'location': 'RER',
        'exchanges': [
            {'input': ('ecoinvent-3.9.1-cutoff', 'transport-lorry'), 'amount': 1.0, 'unit': 'ton kilometer', 'type': 'production'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 'amount': 0.165, 'unit': 'kilogram', 'type': 'biosphere'},
        ]
    },
    ('ecoinvent-3.9.1-cutoff', 'electricity-mv'): {
        'name': 'market for electricity, medium voltage',
        'reference product': 'electricity, medium voltage',
        'unit': 'kilowatt hour',
        'location': 'Europe without Switzerland',
        'exchanges': [
            {'input': ('ecoinvent-3.9.1-cutoff', 'electricity-mv'), 'amount': 1.0, 'unit': 'kilowatt hour', 'type': 'production'},
            {'input': ('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 'amount': 0.385, 'unit': 'kilogram', 'type': 'biosphere'},
        ]
    }
}
bg_db.write(cutoff_data)
print(f"[数据库构建] 背景过程库已写入: {bg_db.name} (包含 {len(bg_db)} 个背景产业链工序)")

# 3.3 构建 4 类国际权威环境影响评价方法 (LCIA Methods)
# -------------------------------------------------------------
method_gwp = ('EF v3.0', 'climate change', 'global warming potential (GWP100)')
m_gwp = bd.Method(method_gwp)
m_gwp.write([
    (('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 1.0),
    (('ecoinvent-3.9.1-biosphere', 'ch4-fossil'), 29.8),
    (('ecoinvent-3.9.1-biosphere', 'n2o'), 273.0),
])

method_ipcc = ('IPCC 2021', 'climate change', 'global warming potential (GWP100)')
m_ipcc = bd.Method(method_ipcc)
m_ipcc.write([
    (('ecoinvent-3.9.1-biosphere', 'co2-fossil'), 1.0),
    (('ecoinvent-3.9.1-biosphere', 'ch4-fossil'), 27.2),
    (('ecoinvent-3.9.1-biosphere', 'n2o'), 273.0),
])

method_water = ('EF v3.0', 'water use', 'water consumption')
m_water = bd.Method(method_water)
m_water.write([
    (('ecoinvent-3.9.1-biosphere', 'water-well'), 42.2),
])

method_acidi = ('EF v3.0', 'acidification', 'accumulated exceedance (AE)')
m_acidi = bd.Method(method_acidi)
m_acidi.write([
    (('ecoinvent-3.9.1-biosphere', 'so2'), 1.31),
])

print(f"[方法注册] 成功注册 4 组 LCIA 特征化方法:")
for m in [method_gwp, method_ipcc, method_water, method_acidi]:
    print(f"   * {m}")


---

## 第 4 步：前景清单 Excel 导入与策略匹配

在本章中，我们将通过 Brightway 官方标准的 `ExcelImporter`，将存放在 `data_collections/excel_importer_example.xlsx` 的前景生命周期清单加载到内存中。

### 匹配与策略流水线 (Matching Pipeline)
1. **`imp.apply_strategies()`**：执行内置标准化规则（单位归一化、去除多余字符、生成 Activity Hash 唯一标识等）。
2. **`imp.match_database("ecoinvent-3.9.1-cutoff", ...)`**：将技术流输入与背景数据库中的工序建立链接。
3. **`imp.match_database("ecoinvent-3.9.1-biosphere", ...)`**：将环境直接排放流与环境基本流数据库建立链接。
4. **`imp.match_database(...)`**：实现前景工序之间的内部互联（例如：饮用水活动消费塑料水瓶制造工序的产品）。
5. **`imp.write_database()`**：当且仅当所有流均建立有效链接（`all_linked == True`）后，写入当前项目。


In [ ]:
# =============================================================
# 第 4 步：导入标准前景 Excel 清单并建立图链接
# =============================================================
excel_path = DATA_DIR / "excel_importer_example.xlsx"
print(f"正在加载 Excel 文件: {excel_path} ...")

imp = bi.ExcelImporter(excel_path)

# 4.1 应用数据预处理与标准化策略流水线
imp.apply_strategies()

# 4.2 跨数据库多维字段匹配 (按 name, unit, location)
imp.match_database("ecoinvent-3.9.1-cutoff", fields=('name', 'unit', 'location'))
imp.match_database("ecoinvent-3.9.1-biosphere", fields=('name', 'unit', 'categories'))
imp.match_database(fields=('name', 'unit', 'location')) # 内部前景工序间匹配

# 4.3 查看匹配统计信息
imp.statistics()

# 4.4 校验链接状态并写入数据库
# 在 BW2.5 中，imp.all_linked 是属性 (布尔值)
if imp.all_linked:
    print("[匹配校验] 所有清单交换流已 100% 成功链接！")
    imp.write_database()
    print(f"[入库成功] 成功写入前景数据库: water_bottles")
else:
    print(f"[匹配警告] 仍存在未链接流 ({len(list(imp.unlinked))} 项)，请检查数据！")

# 4.5 打印数据库中已注册的前景工序与输入清单
wb_db = bd.Database("water_bottles")
print(f"\n[清单核查] [water_bottles] 数据库节点清单:")
for act in wb_db:
    print(f"   * 活动名称: {act['name']:<30} | 产出单位: {act['unit']:<8} | 区域: {act['location']}")
    for exc in act.exchanges():
        print(f"     └─ [{exc['type']:<12}] {exc.input['name']} ({exc['amount']} {exc['unit']})")


---

## 第 5 步：教学专题：未链接流排错与修复演练 (Troubleshooting Unlinked Flows)

> **工业界与学术界最常见痛点**：
> 在 LCA 清单建模实践中，最常遇到的错误是**流未链接（Unlinked Exchanges）**。例如由于单位拼写不一致（`piece` vs `unit`）、活动名称存在空格、或者地理位置代码不匹配，导致系统无法自动在背景数据库中定位到对应的上游活动。
>
> 官方教程特意提供了含有错误的前景文件 `excel_importer_example_w_unlinked.xlsx`。在此环节，我们将体验：
> 1. 如何通过 `imp.statistics()` 捕捉未匹配异常；
> 2. 如何提取 `imp.unlinked` 清单进行精准原因定位；
> 3. 如何在 Python 内存中编写代码**动态修复数据**并恢复 100% 匹配入库。


In [ ]:
# =============================================================
# 第 5 步：教学排错演练：含有未匹配流的数据集诊断与动态修复
# =============================================================
bad_excel_path = DATA_DIR / "excel_importer_example_w_unlinked.xlsx"
print(f"正在加载含有未匹配项的测试清单: {bad_excel_path.name} ...")

imp_bad = bi.ExcelImporter(bad_excel_path)
imp_bad.apply_strategies()

# 尝试执行常规匹配
imp_bad.match_database("ecoinvent-3.9.1-cutoff", fields=('name', 'unit', 'location'))
imp_bad.match_database("ecoinvent-3.9.1-biosphere", fields=('name', 'unit', 'categories'))
imp_bad.match_database(fields=('name', 'unit', 'location'))

print("\n--- 匹配统计结果 ---")
imp_bad.statistics()

# 5.1 深入诊断未匹配原因
# -------------------------------------------------------------
print(f"\n[排错诊断] 当前 all_linked 状态: {imp_bad.all_linked}")
unlinked_list = list(imp_bad.unlinked)
print(f"[排错诊断] 未链接流详细信息清单 (共 {len(unlinked_list)} 项):")

unlinked_records = []
for u in unlinked_list:
    unlinked_records.append({
        "流名称 (name)": u.get('name'),
        "待匹配单位 (unit)": u.get('unit'),
        "目标数据库 (database)": u.get('database'),
        "区域代码 (location)": u.get('location'),
        "流类型 (type)": u.get('type')
    })
df_unlinked = pd.DataFrame(unlinked_records)
print(df_unlinked.to_string())

print("\n[排错根因分析]:")
print("   在上游生产工序中，'water bottle production' 定义的基准产品单位为 'unit'；")
print("   但在 'Drinking water from a bottle' 的清单输入中，将该依赖项单位误写成了 'piece'！")
print("   因 'piece' != 'unit'，默认字段对齐策略判定匹配失败！\n")

# 5.2 动态修复演示：在 Python 内存中校正字段
# -------------------------------------------------------------
print("[排错修复] 正在执行内存补丁动态修复单位...")
fixed_count = 0
for act_data in imp_bad.data:
    for exc in act_data.get('exchanges', []):
        if exc.get('name') == 'water bottle production' and exc.get('unit') == 'piece':
            exc['unit'] = 'unit'
            fixed_count += 1
            print(f"   * 已成功修正节点 [{act_data['name']}] 中的输入流单位: 'piece' -> 'unit'")

# 再次重新触发匹配
imp_bad.match_database(fields=('name', 'unit', 'location'))
print(f"\n[排错结果] 修复后 all_linked 状态: {imp_bad.all_linked}")
print(f"[排错结果] 剩余未链接流数量: {len(list(imp_bad.unlinked))}")
assert imp_bad.all_linked, "修复后应全部成功链接！"
print("[排错完成] 故障排除教学演练圆满完成！")


---

## 第 6 步：单工序 LCIA 计算与 BW2.5 核心 API 剖析

### BW2.5 的计算范式革新：`prepare_lca_inputs`
在 Brightway 2 中，我们通常直接实例化 `LCA({act: 1}, method)`，由系统自动从 SQLite 数据库提取数据并就地求解。

在 **Brightway 2.5** 中，计算流程被高度模块化：
1. **`bd.prepare_lca_inputs(demand, method=...)`**：负责解析用户需求（Functional Unit）并收集所有必要的底层矩阵数据包（Data Objects），返回规范化的数据结构。
2. **`bc.LCA(demand=..., data_objs=...)`**：接收标准化的 `data_objs`，驱动高性能线性代数求解器。这使得计算过程可以完全脱离关系型数据库，极大增强了云原生与批量计算的敏捷度！


In [ ]:
# =============================================================
# 第 6 步：单工序 LCIA 评价计算 (以 1 升瓶装饮用水为例)
# =============================================================

# 6.1 获取目标评价活动与环境影响评价方法
act_water = bd.Database("water_bottles").get(name="Drinking water from a bottle")
print(f"[目标活动] 选定功能单位活动: {act_water}")
print(f"[评价方法] 选定评价方法:     {method_gwp}")

# 6.2 使用 BW2.5 新 API 准备计算输入对象
# prepare_lca_inputs 会自动组装技术矩阵、基本流矩阵与特征化矩阵的数据包
my_demand, data_objs, remapping = bd.prepare_lca_inputs({act_water: 1}, method=method_gwp)

print(f"\n[数据包] 组装生成的 Data Objects 数量: {len(data_objs)}")
for i, obj in enumerate(data_objs):
    print(f"   [{i+1}] {type(obj).__name__}")

# 6.3 实例化 LCA 求解器并执行计算
my_lca = bc.LCA(demand=my_demand, data_objs=data_objs)
my_lca.lci()  # 清单分析 (求解 A·s = d)
my_lca.lcia() # 影响评价 (求解 h = C·B·s)

# 6.4 输出关键环境影响分值
print(f"\n[计算成果] 1 升瓶装水生命周期碳足迹 (GWP100):")
print(f"   >>> {my_lca.score:.6f} kg CO2-eq / liter")
print(f"   (包含 1L 自来水提取净化 + 1个PET塑料瓶注塑成型 + 电力能耗 + 货运物流全程排放)")


---

## 第 7 步：全新 2024 MultiLCA 批量计算与多指标对比分析

在实际科研与工程咨询中，我们往往需要同时评估**多个产品方案**在**多种环境影响类别（如气候变化、水资源利用、酸化潜力等）**下的综合表现。

### BW2.5 全新 `MultiLCA` 机制 (2024 Release)
- **多功能单位定义**：支持以字典形式同时定义多个产品情景；
- **`get_multilca_data_objs`**：单次解析生成跨所有活动与评价方法的超维数据对象；
- **矩阵并行运算**：在底层直接完成多列需求向量与多组特征化因子的矩阵乘法，计算效率相比传统串行循环提升一个数量级！


In [ ]:
# =============================================================
# 第 7 步：全新 2024 MultiLCA 批量计算与多指标分析
# =============================================================
act_bottle = bd.Database("water_bottles").get(name="water bottle production")

# 7.1 定义多个功能单位需求 (情景 1: 1L 瓶装饮用水 vs 情景 2: 仅塑料水瓶包装制造)
functional_units = {
    "1L 瓶装水全生命周期": {act_water.id: 1},
    "单个塑料水瓶包装制备": {act_bottle.id: 1}
}

# 7.2 定义评价方法集合
methods_list = [method_gwp, method_ipcc, method_water, method_acidi]
method_config = {"impact_categories": methods_list}

# 7.3 构建 MultiLCA 数据包
mlca_data_objs = bd.get_multilca_data_objs(
    functional_units=functional_units,
    method_config=method_config
)

# 7.4 执行 MultiLCA 批量求解
mlca = bc.MultiLCA(
    demands=functional_units,
    method_config=method_config,
    data_objs=mlca_data_objs
)
mlca.lci()
mlca.lcia()

# 7.5 将多维计算结果整理为学术级 Pandas DataFrame
records = []
for (method_tuple, act_name), val in mlca.scores.items():
    cat_name = method_tuple[1]
    metric_name = method_tuple[2]
    records.append({
        "评价体系": method_tuple[0],
        "环境影响类别": f"{cat_name} ({metric_name})",
        "产品工序方案": act_name,
        "环境影响数值": val
    })

df_mlca_raw = pd.DataFrame(records)
df_mlca = df_mlca_raw.pivot(index=["评价体系", "环境影响类别"], columns="产品工序方案", values="环境影响数值")
print("\n[MultiLCA 结果汇总表]:")
print(df_mlca.to_string())

# 7.6 绘图对比：主要环境影响类别指标柱状对比图
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Source Han Sans CN', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图 1: 气候变化与碳足迹对比 (kg CO2-eq)
gwp_data = df_mlca_raw[df_mlca_raw["环境影响类别"].str.contains("climate change")].copy()
sns.barplot(data=gwp_data, x="评价体系", y="环境影响数值", hue="产品工序方案", ax=axes[0], palette="Blues_d")
axes[0].set_title("图 7.1 碳足迹对比: 1L 瓶装水 vs 塑料水瓶 (kg CO2-eq)", fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel("kg CO2-eq", fontsize=11)
axes[0].set_xlabel("评价标准", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.6)

# 子图 2: 水耗与酸化潜能对比
other_data = df_mlca_raw[~df_mlca_raw["环境影响类别"].str.contains("climate change")].copy()
sns.barplot(data=other_data, x="环境影响类别", y="环境影响数值", hue="产品工序方案", ax=axes[1], palette="Greens_d")
axes[1].set_title("图 7.2 资源利用与环境排放潜能对比", fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylabel("指标量纲分值", fontsize=11)
axes[1].set_xlabel("影响类别", fontsize=11)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


---

## 第 8 步：底层矩阵解构、贡献度溯源与专业级多 Sheet Excel 报告导出

### 8.1 底层矩阵结构剖析
Brightway 2.5 继承并深化了 Heijungs 与 Suh 的生命周期评价矩阵代数理论：
$$\mathbf{A} \cdot \mathbf{s} = \mathbf{d} \implies \mathbf{s} = \mathbf{A}^{-1} \cdot \mathbf{d}$$
$$\mathbf{g} = \mathbf{B} \cdot \mathbf{s}$$
$$h = \mathbf{c}^T \cdot \mathbf{g} = \mathbf{c}^T \cdot \mathbf{B} \cdot \mathbf{A}^{-1} \cdot \mathbf{d}$$

在 BW2.5 中，我们可以通过 `lca.technosphere_matrix`（技术矩阵 $\mathbf{A}$）、`lca.biosphere_matrix`（基本流矩阵 $\mathbf{B}$）及 `lca.characterization_matrix`（特征化矩阵 $\mathbf{C}$）直接进行底层数据探索！

### 8.2 贡献度分析与工序溯源 (`bw2analyzer`)
借助 `bw2analyzer.ContributionAnalysis` 与 `print_recursive_calculation`，我们将完成：
1. 顶层贡献过程拆解（Top Contributing Processes）
2. 关键环境排放流识别（Top Contributing Biosphere Flows）
3. 递归树形层级溯源
4. 包装轻量化与再生塑料（rPET）脱碳情景敏感性分析
5. 自动化导出 4 工作表的专业 Excel 数据报告 (`lcia_results_practice2.xlsx`)


In [ ]:
# =============================================================
# 第 8 步：底层矩阵解构、贡献度分析与多工作表 Excel 导出
# =============================================================

# 8.1 探索底层科学计算矩阵维度与非零稀疏元
# -------------------------------------------------------------
print("[矩阵解构] BW2.5 底层矩阵特征剖析:")
print(f"   * 技术矩阵 (Technosphere A):       维度 {my_lca.technosphere_matrix.shape} | 非零元数: {my_lca.technosphere_matrix.nnz}")
print(f"   * 基本流矩阵 (Biosphere B):         维度 {my_lca.biosphere_matrix.shape} | 非零元数: {my_lca.biosphere_matrix.nnz}")
print(f"   * 特征化矩阵 (Characterization C): 维度 {my_lca.characterization_matrix.shape} | 非零元数: {my_lca.characterization_matrix.nnz}")
print(f"   * 供应链求解向量 (Supply vector s): 长度 {len(my_lca.supply_array)}")
print(f"   * 特征化清单总量核验:              {my_lca.characterized_inventory.sum():.6f} kg CO2-eq")

# 8.2 使用 bw2analyzer 执行贡献度溯源
# -------------------------------------------------------------
ca = bwa.ContributionAnalysis()
top_processes = ca.annotated_top_processes(my_lca, limit=6)
top_emissions = ca.annotated_top_emissions(my_lca, limit=5)

proc_records = []
for score, supply, act in top_processes:
    pct = (score / my_lca.score) * 100
    proc_records.append({
        "工序名称": act['name'],
        "绝对贡献 (kg CO2-eq)": score,
        "贡献比例 (%)": pct,
        "供应链产出量": supply,
        "单位": act['unit'],
        "所属数据库": act['database']
    })
df_top_processes = pd.DataFrame(proc_records)

print("\n[贡献排序] 工序贡献度排序 (Top Processes):")
print(df_top_processes[["工序名称", "绝对贡献 (kg CO2-eq)", "贡献比例 (%)", "供应链产出量", "单位"]].to_string())

# 8.3 递归树形计算打印 (官方同款 API)
# -------------------------------------------------------------
print("\n[树状溯源] 递归生命周期供应链树状贡献度拆解 (Recursive Breakdown):")
bwa.print_recursive_calculation(act_water, method_gwp, cutoff=0.01)

# 8.4 可视化分析：瓶装水碳足迹结构与贡献度瀑布拆解
# -------------------------------------------------------------
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Source Han Sans CN', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# 图 8.1 各工序碳排放贡献占比饼图
labels = [row["工序名称"][:25] + "..." if len(row["工序名称"]) > 25 else row["工序名称"] for _, row in df_top_processes.iterrows()]
colors = sns.color_palette("mako", len(labels))
ax1.pie(df_top_processes["贡献比例 (%)"], labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)
ax1.set_title("图 8.1 瓶装饮用水生命周期碳足迹环节构成", fontsize=13, fontweight='bold', pad=12)

# 图 8.2 塑料瓶减重与再生料 (rPET) 脱碳情景分析
# 情景: 1.基准 (0% rPET, 32g) | 2.瓶身轻量化20% (25.6g) | 3.添加50% rPET (减碳40%) | 4.轻量化+70% rPET
scenarios = {
    "基准方案 (100%原生PET, 32g)": my_lca.score,
    "轻量化 20% (瓶重 25.6g)": my_lca.score - 0.20 * (0.073568 + 0.0272),
    "50% 再生料 rPET 替代": my_lca.score - 0.50 * 0.073568 * 0.65,
    "轻量化20% + 70% rPET": my_lca.score - (0.20 * 0.0272 + (1 - 0.8*0.3) * 0.073568 * 0.65)
}
df_scenarios = pd.DataFrame(list(scenarios.items()), columns=["情景设计", "碳足迹 (kg CO2-eq)"])
df_scenarios["相对基准降碳比例 (%)"] = (1 - df_scenarios["碳足迹 (kg CO2-eq)"] / my_lca.score) * 100

sns.barplot(data=df_scenarios, x="相对基准降碳比例 (%)", y="情景设计", ax=ax2, palette="viridis")
ax2.set_title("图 8.2 包装工程改进情景脱碳潜力敏感性分析", fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel("相较基准降碳比例 (%)", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

# 8.5 导出教学成果至包含 4 工作表的专业 Excel 数据表
# -------------------------------------------------------------
output_excel_path = Path("lcia_results_practice2.xlsx")
with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
    # Sheet 1: 多指标方案概览
    df_mlca_raw.to_excel(writer, sheet_name="1_MultiLCA_Overview", index=False)
    # Sheet 2: 工序贡献度明细
    df_top_processes.to_excel(writer, sheet_name="2_Process_Contributions", index=False)
    # Sheet 3: 脱碳情景敏感性分析
    df_scenarios.to_excel(writer, sheet_name="3_Decarbonization_Scenarios", index=False)
    # Sheet 4: 关键基本流排放明细
    top_emiss_df = pd.DataFrame([
        {"基本流": flow['name'], "排放量": inv, "特征化碳得分": sc, "单位": flow['unit']}
        for sc, inv, flow in top_emissions
    ])
    top_emiss_df.to_excel(writer, sheet_name="4_Key_Emissions", index=False)

print(f"\n[成果交付] 成功导出综合评价成果至 Excel: {output_excel_path.resolve()}")
print(f"   * 工作表 1: 1_MultiLCA_Overview (多指标对比)")
print(f"   * 工作表 2: 2_Process_Contributions (工序贡献度排序)")
print(f"   * 工作表 3: 3_Decarbonization_Scenarios (脱碳敏感性情景)")
print(f"   * 工作表 4: 4_Key_Emissions (关键环境排放清单)")


---

## 第 9 步：蒙特卡洛不确定性分析与分布统计可视化

在工程 LCA 实践中，由于生产批次波动、设备老化、测量误差等原因，清单数据往往存在显著的不确定性。

### BW2.5 蒙特卡洛计算机制
- 在定义工序交换流时，可声明其不确定性分布参数（如对数正态分布 `uncertainty type: 2`、三角分布 `uncertainty type: 5` 等）；
- 在初始化 `bc.LCA` 时传入参数 `use_distributions=True`；
- 调用 Python 标准迭代协议 `next(mc_lca)`，求解器将自动重新抽样并更新稀疏矩阵中的参数，快速完成成百上千次随机抽样运算！


In [ ]:
# =============================================================
# 第 9 步：蒙特卡洛不确定性分析 (500 次随机抽样模拟)
# =============================================================

# 9.1 动态为前景与背景工序注入真实工程不确定性参数
# 将 PET 树脂原料单耗设为对数正态分布 (变异系数 CV=12%)
for exc in act_bottle.exchanges():
    if "polyethylene terephthalate" in exc.input['name']:
        exc['uncertainty type'] = 2 # Lognormal
        exc['loc'] = float(np.log(exc['amount']))
        exc['scale'] = 0.12 # 几何标准差波动
        exc.save()

# 9.2 重新生成包含不确定性分布的数据包
mc_demand, mc_data_objs, _ = bd.prepare_lca_inputs({act_water: 1}, method=method_gwp)

# 9.3 实例化支持不确定性抽样的 LCA 求解器
mc_lca = bc.LCA(demand=mc_demand, data_objs=mc_data_objs, use_distributions=True)
mc_lca.lci()
mc_lca.lcia()

# 9.4 执行 500 次蒙特卡洛抽样计算
N_RUNS = 500
print(f"[蒙特卡洛] 正在启动蒙特卡洛模拟 (迭代次数 N={N_RUNS})，请稍候...")

mc_scores = []
for i in range(N_RUNS):
    next(mc_lca)
    mc_scores.append(mc_lca.score)

mc_scores = np.array(mc_scores)
print("[蒙特卡洛] 抽样模拟顺利完成！")

# 9.5 计算描述性统计学指标
mean_score = np.mean(mc_scores)
std_score = np.std(mc_scores)
median_score = np.median(mc_scores)
ci_lower, ci_upper = np.percentile(mc_scores, [2.5, 97.5])
p5, p95 = np.percentile(mc_scores, [5.0, 95.0])

print(f"\n[统计报告] 蒙特卡洛统计学指标报告:")
print(f"   * 样本均值 (Mean):        {mean_score:.6f} kg CO2-eq")
print(f"   * 标准差 (Std Dev):       {std_score:.6f} kg CO2-eq")
print(f"   * 中位数 (P50 Median):    {median_score:.6f} kg CO2-eq")
print(f"   * 95% 置信区间 (95% CI):   [{ci_lower:.6f}, {ci_upper:.6f}]")
print(f"   * 5%~95% 分位数范围:      [{p5:.6f}, {p95:.6f}]")

# 9.6 绘制出版级概率密度分布直方图与 KDE 曲线
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Source Han Sans CN', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.figure(figsize=(10, 5.5))
sns.histplot(mc_scores, bins=35, kde=True, color="#2b5c8f", stat="density", alpha=0.45, edgecolor="white")

plt.axvline(mean_score, color="#d9534f", linestyle="--", linewidth=2, label=f"均值: {mean_score:.4f}")
plt.axvline(median_score, color="#5cb85c", linestyle=":", linewidth=2, label=f"中位数: {median_score:.4f}")
plt.axvspan(ci_lower, ci_upper, color="#f0ad4e", alpha=0.18, label=f"95% 置信区间: [{ci_lower:.3f}, {ci_upper:.3f}]")

plt.title("图 9.1 瓶装水生命周期碳足迹蒙特卡洛概率密度分布 (N=500)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("生命周期碳排放 (kg CO2-eq / 1L 瓶装水)", fontsize=11)
plt.ylabel("概率密度 (Density)", fontsize=11)
plt.legend(frameon=True, facecolor="white", edgecolor="none")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# 9.7 将蒙特卡洛样本数据追加至 Excel 报告 Sheet 5
with pd.ExcelWriter(output_excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_mc = pd.DataFrame({
        "抽样序号": np.arange(1, N_RUNS + 1),
        "单次模拟得分 (kg CO2-eq)": mc_scores
    })
    df_mc.to_excel(writer, sheet_name="5_Monte_Carlo_Distribution", index=False)

print(f"[报告追加] 蒙特卡洛样本数据已成功追加写入至: {output_excel_path.name} -> Sheet: 5_Monte_Carlo_Distribution")
